## 1. Setup und Imports

Libraries importieren, äußere Form festlegen und Output-Ordner erstellen

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns
import geopandas as gpd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.preprocessing import LabelEncoder
from scipy import stats

os.environ['SHAPE_RESTORE_SHX'] = 'YES'

# --- Design-System ---
PALETTE_MAIN   = '#1a3a5c'
PALETTE_ACC    = '#e07b39'
PALETTE_NEG    = '#c0392b'
PALETTE_POS    = '#27ae60'
PALETTE_NEUTRAL= '#7f8c8d'
CMAP_CHOROPLETH = 'YlOrRd'
CMAP_DIVERG     = 'RdYlGn_r'
COLOR_SEQ = ['#1a3a5c','#2e6da4','#e07b39','#e8a87c','#27ae60','#c0392b','#8e44ad','#f39c12']

sns.set_theme(style='whitegrid', context='talk', font_scale=0.95)
plt.rcParams.update({
    'figure.figsize': (14, 8), 'figure.dpi': 130, 'savefig.dpi': 180,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titleweight': 'bold', 'axes.titlesize': 14,
    'axes.labelweight': 'regular', 'axes.labelsize': 12,
    'axes.grid': True, 'grid.alpha': 0.20, 'grid.linestyle': '--',
    'legend.frameon': False,
    'axes.prop_cycle': plt.cycler(color=COLOR_SEQ),
})

OUTPUT_DIR = '/home/mike/Documents/Economic Data Science/Case Study/output/figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def savefig(name):
    plt.savefig(f'{OUTPUT_DIR}/{name}.png', bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'  -> gespeichert: {name}.png')

## 2. Datenladen

Die Hauptdatei enthält rund 200.000 Mietinserate aus Dortmund. Zusätzlich werden PLZ-Gemeinde-Schlüssel, Stadtteilzuordnungen und Geodaten geladen.

In [ ]:
BASE = '/home/mike/Documents/Economic Data Science/Case Study/data'

df          = pd.read_csv(f'{BASE}/wohnungsmieten_dortmund.csv', low_memory=False)
plz_ort     = pd.read_csv(f'{BASE}/zuordnung_plz_ort.csv')
stadtteile  = pd.read_csv(f'{BASE}/dortmund_stadtteile_plz_onlinestreet.csv')
avg_plz     = pd.read_csv(f'{BASE}/dortmund_avg_plz.csv')

plz_ort['plz'] = plz_ort['plz'].astype(str)

print(f'Rohdaten: {df.shape[0]:,} Zeilen, {df.shape[1]} Spalten')
df.head(3)


## 3. Datenbereinigung

### Bekannte Datenproblem

- `baujahr` fehlt bei ~37% der Inserate (ältere/schlechter dokumentierte Objekte)
- `ev_kennwert` (Energiekennwert) fehlt bei ~65%
- `heizkosten` fehlt bei ~57%, kann aber wenn "ja" in `heizkosten_in_wm_enthalten` schon in der Warmmiete (mietekalt + nebenkosten) enthalten sein
- Mehrere Spalten sind zum Großteil mit `Other missing` oder ähnlichen Variablen belegt


### Bereinigungsschritte
1. Typkorrektur aller numerischen Felder
2. Zimmeranzahl: Nur valide Werte (Vielfache von 0.5, > 0)
3. Outlier-Filter für `rent_sqm` (0.5%–99.5%-Quantil)
4. Wohnfläche: Plausibilitätsbereich 15–200 m²

In [ ]:
list(df)

In [ ]:
# Typkorrektur
df['plz']     = df['plz'].astype(str).str.strip()
df['baujahr'] = pd.to_numeric(df['baujahr'], errors='coerce')

# Zimmeranzahl bereinigen
df['zimmeranzahl'] = pd.to_numeric(df['zimmeranzahl'], errors='coerce')
mask = df['zimmeranzahl'].notna() & (df['zimmeranzahl'] > 0) & ((df['zimmeranzahl'] * 2) % 1 == 0)
df = df[mask].copy()

# Numerische Felder
for col in ['heizkosten', 'nebenkosten', 'mietekalt', 'wohnflaeche']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Ausreißer-Filter (IQR-basiert)
q_lo = df['rent_sqm'].quantile(0.005)
q_hi = df['rent_sqm'].quantile(0.995)
df = df[(df['rent_sqm'] >= q_lo) & (df['rent_sqm'] <= q_hi)].copy()
df = df[(df['wohnflaeche'] >= 15) & (df['wohnflaeche'] <= 200)].copy()

print(f'Nach Bereinigung: {df.shape[0]:,} Zeilen')

In [ ]:
# Fehlende-Werte-Übersicht
check_cols = ['rent_sqm','wohnflaeche','baujahr','mietekalt','heizkosten',
              'zimmeranzahl','heizungsart','objektzustand','ausstattung',
              'energieeffizienzklasse','ev_kennwert', 'heizkosten_in_wm_enthalten', 'nebenkosten']

missing = pd.DataFrame({
    'Fehlende Werte': df[check_cols].isna().sum(),
    'Anteil (%)': (df[check_cols].isna().mean() * 100).round(1)
}).sort_values('Fehlende Werte', ascending=False)
missing

## 4. Merge – PLZ, AGS und Stadtteil


In [ ]:
# Nur Dortmund-Zeilen aus plz_ort – dedupliziert
plz_ort_do = plz_ort[plz_ort['ort'] == 'Dortmund'][['ags','plz']].copy()
plz_ort_do['plz'] = plz_ort_do['plz'].astype(str)
plz_ort_do = plz_ort_do.drop_duplicates(subset='plz') 

df = df.merge(plz_ort_do[['plz','ags']], on='plz', how='left')

# Stadtteil-Merge: erstes Auftreten je PLZ = dominanter Stadtteil
stadtteile['plz'] = stadtteile['plz'].astype(str)
stadtteile_dedup = stadtteile.drop_duplicates(subset='plz', keep='first')
df = df.merge(stadtteile_dedup, on='plz', how='left')

print(f'df nach Merge: {df.shape[0]:,} Zeilen, {df.shape[1]} Spalten')
print(f'Stadtteil zugeordnet: {df['stadtteil'].notna().sum():,} Inserate')

In [ ]:
df

In [ ]:
max_wert = df["edat_dt"].max()
max_wert

In [ ]:
min_wert = df["edat_dt"].min()
min_wert

## 5. Feature Engineering

- **Gebäudealter** und **Bauepoche** aus `baujahr`
- **Modernisierungsstatus** (Dummy + Jahre seit letzter Modernisierung)
- **Ausstattungs-Ordinalskala** (Simple=1 bis Deluxe=4)
- **Energieeffizienz-Ordinalskala** (A+=1 bis H=9)
- **Objektzustands-Ordinalskala** (Abrissreif=0 bis Erstbezug=9)
- **Kosten je m²** für Heiz- und Nebenkosten
- **Binäre Ausstattungsmerkmale** (Balkon, Aufzug, Einbauküche, etc.)

In [ ]:
# Modernisierung
df['mod_jahr']         = pd.to_numeric(df['letzte_modernisierung'], errors='coerce')
df['wurde_modernisiert'] = df['mod_jahr'].notna().astype(int)
df['jahre_seit_mod']    = np.where(df['mod_jahr'].notna(), 2022 - df['mod_jahr'], np.nan)

# Gebäudealter
df['gebaeudealter'] = np.where(df['baujahr'].notna(), 2022 - df['baujahr'], np.nan)

# Bauepoche
def bauepoche(j):
    if pd.isna(j): return 'Unbekannt'
    if j < 1920:   return 'Industrialisierung (<1919)'
    if j < 1940:   return 'Zwischenkriegszeit (1919–39)'
    if j < 1960:   return '(Nach-)kriegszeit (1940-59)'
    if j < 1990:   return 'Kohle und Stahlkrise (1960–89)'
    if j < 2010:   return 'Post-industrieller Strukturwandel (90–2009)'
    return 'Modern (2009+)'
df['bauepoche'] = df['baujahr'].apply(bauepoche)

# Ordinale Skalen
ausstattung_map = {'Simple':1,'Normal':2,'Sophisticated':3,'Deluxe':4}
df['ausstattung_num'] = df['ausstattung'].map(ausstattung_map)

eff_map = {'APLUS':1,'A':2,'B':3,'C':4,'D':5,'E':6,'F':7,'G':8,'H':9}
df['eeff_num'] = df['energieeffizienzklasse'].map(eff_map)

df['nk_sqm']    = df['nebenkosten'] / df['wohnflaeche']
df['heizk_sqm'] = df['heizkosten']  / df['wohnflaeche']

# Binäre Dummies
bool_cols = ['aufzug','balkon','einbaukueche','keller','gaestewc','garten','parkplatz']
for col in bool_cols:
    df[col+'_bin'] = (df[col] == 'Yes').astype(int)

# Deutsche Labels
heizung_map = {
    'Central heating':'Zentralheizung', 'Self-contained central heating':'Etagenheizung',
    'Gas heating':'Gasheizung', 'Floor heating':'Fußbodenheizung',
    'District heating':'Fernwärme', 'Oil heating':'Ölheizung',
    'Night storage heaters':'Nachtspeicher', 'Thermal heat pump':'Wärmepumpe',
    'Electric heating':'Elektroheizung', 'Not specified':'Nicht angegeben'
}
df['heizungsart_de'] = df['heizungsart'].map(heizung_map).fillna('Sonstiges')

zustand_map = {
    'First occupancy':'Erstbezug', 'First occupancy after reconstruction':'Erstbezug nach Sanierung',
    'Like new':'Wie neu', 'Modernised':'Modernisiert', 'Completely renovated':'Vollsaniert',
    'Reconstructed':'Saniert', 'Well kempt':'Gepflegt', 'By arrangement':'Nach Vereinbarung',
    'Needs renovation':'Renovierungsbedarf', 'Dilapidated':'Abrissreif', 'Not specified':'Nicht angegeben'
}
df['objektzustand_de'] = df['objektzustand'].map(zustand_map).fillna('Nicht angegeben')
zustand_order = {
    'Abrissreif':0,'Renovierungsbedarf':1,'Nach Vereinbarung':2,'Gepflegt':3,'Saniert':4,
    'Vollsaniert':5,'Modernisiert':6,'Wie neu':7,'Erstbezug nach Sanierung':8,'Erstbezug':9,'Nicht angegeben':np.nan
}
df['zustand_num'] = df['objektzustand_de'].map(zustand_order)

print('Feature Engineering abgeschlossen.')
print(f'Datensatz hat jetzt {df.shape[1]} Spalten.')

## 6. Deskriptive Statistik

Mit rund 198.000 Inseraten aus 27 Postleitzahlbezirken deckt der Datensatz den gesamten Dortmunder Mietwohnungsmarkt über einen Zeitraum von mehr als einem Jahrzehnt ab. Der mittlere Quadratmeterpreis liegt bei 6,38 €, der Median bei 6,00 €.

In [ ]:
df[['rent_sqm','wohnflaeche','mietekalt','zimmeranzahl','baujahr']].describe().round(2)

In [ ]:
# PLZ-Tabelle
plz_table = (
    df.groupby('plz', dropna=False)
    .agg(
        Beobachtungen=('obid','size'),
        qmPreis_Mittel=('rent_sqm','mean'),
        qmPreis_Median=('rent_sqm','median'),
        Q25=('rent_sqm', lambda x: x.quantile(0.25)),
        Q75=('rent_sqm', lambda x: x.quantile(0.75)),
        Flaeche_Mittel=('wohnflaeche','mean'),
        Baujahr_Mittel=('baujahr','mean'),
    )
    .sort_values('qmPreis_Mittel', ascending=False)
    .round(2)
)
plz_table

## 7. Visualisierungen

### 7.1 Mietpreis- und Flächenverteilung

Die Verteilung der Angebotsmieten ist rechtsschief: ein kleines Segment hochpreisiger Inserate zieht den Mittelwert über den Median. Die Wohnflächen konzentrieren sich um 50–80 m², was dem typischen Dortmunder Altbaubestand entspricht.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ax.hist(df['rent_sqm'], bins=60, color=PALETTE_MAIN, alpha=0.85, edgecolor='white', linewidth=0.3)
ax.axvline(df['rent_sqm'].mean(), color=PALETTE_ACC, linewidth=2, linestyle='--',
           label=f"Mittelwert: {df['rent_sqm'].mean():.2f} €/m²")
ax.axvline(df['rent_sqm'].median(), color=PALETTE_NEG, linewidth=2, linestyle=':',
           label=f"Median: {df['rent_sqm'].median():.2f} €/m²")
ax.set_xlabel('Kaltmiete je m² (€)')
ax.set_ylabel('Anzahl Inserate')
ax.set_title('Verteilung der Angebotsmieten')
ax.legend()

ax = axes[1]
ax.hist(df['wohnflaeche'], bins=60, color=PALETTE_ACC, alpha=0.85, edgecolor='white', linewidth=0.3)
ax.axvline(df['wohnflaeche'].mean(), color=PALETTE_MAIN, linewidth=2, linestyle='--',
           label=f"Mittelwert: {df['wohnflaeche'].mean():.1f} m²")
ax.set_xlabel('Wohnfläche (m²)')
ax.set_ylabel('Anzahl Inserate')
ax.set_title('Verteilung der Wohnflächen')
ax.legend()

fig.suptitle('Deskriptive Verteilung – Dortmunder Mietwohnungsmarkt', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
savefig('01_verteilung_miete_flaeche')

### 7.2 Mietpreise nach Postleitzahl (Boxplot)

Die innenstadtnahen PLZ-Bezirke 44139 und 44229 erzielen die höchsten Preise (Median ~7,3 €/m²), während Randbezirke wie 44328 deutlich darunter liegen (~5,1 €/m²). Die Spannweiten zeigen ausgeprägte Heterogenität auch innerhalb einzelner Bezirke.

In [ ]:
plz_order = df.groupby('plz')['rent_sqm'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(18, 7))
bp_data = [df[df['plz'] == p]['rent_sqm'].dropna().values for p in plz_order]
colors_bp = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(plz_order)))

bp = ax.boxplot(bp_data, patch_artist=True, showfliers=False,
                medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(color); patch.set_alpha(0.85)

ax.set_xticklabels(plz_order, rotation=45, ha='right', fontsize=9)
ax.set_xlabel('Postleitzahl')
ax.set_ylabel('Kaltmiete je m² (€)')
ax.set_title('Mietpreisspannen nach Postleitzahl – sortiert nach Median')
plt.tight_layout()
savefig('02_boxplot_plz')

### 7.3 Choropleth-Karte – Mietpreis und Wohnstruktur

Die räumliche Kartierung zeigt ein deutliches Gefälle von der Stadtmitte (44139, 44135, 44229) zu den nordwestlichen und südöstlichen Randlagen (44328, 44359, 44339). Das zweite Kartenpaar visualisiert die Wohnungsstruktur: kleinere Wohnungen dominieren in den innenstadtnahen Bezirken, größere Flächenangebote finden sich in ruhigeren Wohnlagen wie 44229 oder 44265.

In [ ]:
gdf_plz = gpd.read_file("/home/mike/Documents/Economic Data Science/Data/plz-Daten/plz-5stellig.shp")

print(gdf_plz.columns)
print(gdf_plz.head())
print(gdf_plz.shape)

In [ ]:
gdf_plz

In [ ]:
# Geodaten vorbereiten
plz_stats = df.groupby('plz').agg(
    rent_mean=('rent_sqm','mean'), rent_median=('rent_sqm','median'),
    obs=('obid','size'), mean_zimmer=('zimmeranzahl','mean'),
    mean_flaeche=('wohnflaeche','mean'),
).reset_index()

In [ ]:
plz_stats

In [ ]:
gdf_map = gdf_plz.merge(plz_stats, on="plz", how='left')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
gdf_map.plot(column='rent_mean', cmap='YlOrRd', linewidth=0.5, edgecolor='white',
             ax=ax, legend=True, legend_kwds={'label':'Ø Kaltmiete je m² (€)','shrink':0.7})
for _, row in gdf_map.iterrows():
    if row.geometry is not None and not pd.isna(row['rent_mean']):
        c = row.geometry.centroid
        ax.annotate(f"{row['plz']}\n{row['rent_mean']:.2f}€", xy=(c.x, c.y),
                    ha='center', va='center', fontsize=8,
                    color='white' if row['rent_mean'] > 6.8 else 'black')
ax.set_axis_off()
ax.set_title('Durchschnittlicher Quadratmeterpreis nach Postleitzahlbezirk\nDortmund – Mietwohnungsmarkt',
             fontsize=14, fontweight='bold')
plt.tight_layout()
savefig('03_choropleth_mietpreis')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 9))

gdf_map.plot(column='mean_zimmer', cmap='Blues', linewidth=0.5, edgecolor='white',
             ax=axes[0], legend=True, legend_kwds={'label':'Ø Zimmeranzahl','shrink':0.7})
for _, row in gdf_map.iterrows():
    if row.geometry is not None and not pd.isna(row.get('mean_zimmer')):
        c = row.geometry.centroid
        axes[0].annotate(f"{row['plz']}\n{row['mean_zimmer']:.1f}",
                         xy=(c.x, c.y), ha='center', va='center', fontsize=8, color='white' if row['mean_zimmer'] > 2.9 else 'black')
axes[0].set_axis_off()
axes[0].set_title('Ø Zimmeranzahl nach PLZ\n(Infrastruktur-Indikator)', fontsize=12, fontweight='bold')

gdf_map.plot(column='mean_flaeche', cmap='Greens', linewidth=0.5, edgecolor='white',
             ax=axes[1], legend=True, legend_kwds={'label':'Ø Wohnfläche (m²)','shrink':0.7})
for _, row in gdf_map.iterrows():
    if row.geometry is not None and not pd.isna(row.get('mean_flaeche')):
        c = row.geometry.centroid
        axes[1].annotate(f"{row['plz']}\n{row['mean_flaeche']:.0f}m²",
                         xy=(c.x, c.y), ha='center', va='center', fontsize=8, color='white' if row['mean_flaeche'] > 75 else 'black')
axes[1].set_axis_off()
axes[1].set_title('Ø Wohnfläche nach PLZ\n(Marktstruktur)', fontsize=12, fontweight='bold')

fig.suptitle('Strukturelle Marktcharakteristika nach Postleitzahlbezirk', fontsize=14, fontweight='bold')
plt.tight_layout()
savefig('04_choropleth_zimmer_flaeche')

### 7.4 Heizungsart – Verbreitung und Preiseffekt

Zentralheizung dominiert den Markt (46%), gefolgt von Etagenheizung. Moderne Systeme wie Wärmepumpe und Fußbodenheizung sind mit höheren Mietpreisen korreliert, während Nachtspeicherheizungen das Schlusslicht bilden.

In [ ]:
heiz_counts = df['heizungsart_de'].value_counts()
heiz_relevant = heiz_counts[heiz_counts >= 200].index
df_heiz = df[df['heizungsart_de'].isin(heiz_relevant)].copy()
heiz_means = df_heiz.groupby('heizungsart_de')['rent_sqm'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
counts = heiz_counts[heiz_relevant].reindex(heiz_means.index)
colors_h = [COLOR_SEQ[i % len(COLOR_SEQ)] for i in range(len(heiz_means))]

axes[0].barh(range(len(counts)), counts.values, color=colors_h, alpha=0.85, edgecolor='white')
axes[0].set_yticks(range(len(counts))); axes[0].set_yticklabels(counts.index, fontsize=10)
axes[0].set_xlabel('Anzahl Inserate'); axes[0].set_title('Häufigkeit der Heizungsarten')
for i, v in enumerate(counts.values): axes[0].text(v+200, i, f'{v:,}', va='center', fontsize=9)

axes[1].barh(range(len(heiz_means)), heiz_means.values, color=colors_h, alpha=0.85, edgecolor='white')
axes[1].set_yticks(range(len(heiz_means))); axes[1].set_yticklabels(heiz_means.index, fontsize=10)
axes[1].set_xlabel('Ø Kaltmiete je m² (€)'); axes[1].set_title('Mittlerer Mietpreis nach Heizungsart')
axes[1].axvline(df['rent_sqm'].mean(), color=PALETTE_NEG, linestyle='--', linewidth=1.5, label='Gesamtmittel')
axes[1].legend()
for i,v in enumerate(heiz_means.values): axes[1].text(v+0.02, i, f'{v:.2f}€', va='center', fontsize=9)

fig.suptitle('Heizungsart – Verbreitung und Einfluss auf den Mietpreis', fontsize=18, fontweight='bold')
plt.tight_layout(); savefig('05_heizungsart')

### 7.5 Ausstattungsqualität und Objektzustand

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

ausst_means = (
    df[df['ausstattung'] != 'Not specified']
    .groupby('ausstattung')['rent_sqm'].agg(['mean','count','median'])
    .reindex(['Simple','Normal','Sophisticated','Deluxe']).dropna()
)
colors_a = [plt.cm.Blues(v) for v in [0.3,0.5,0.7,0.9]]
bars = axes[0].bar(ausst_means.index, ausst_means['mean'], color=colors_a, alpha=0.9, edgecolor='white', width=0.6)
axes[0].set_xlabel('Ausstattungsqualität'); axes[0].set_ylabel('Ø Kaltmiete je m² (€)')
axes[0].set_title('Mietpreis nach Ausstattungsqualität')
axes[0].axhline(df['rent_sqm'].mean(), color=PALETTE_ACC, linestyle='--', linewidth=1.5, label='Gesamtmittel')
axes[0].legend()
for bar, (_, row) in zip(bars, ausst_means.iterrows()):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                f"{row['mean']:.2f}€\n(n={row['count']:,.0f})", ha='center', va='bottom', fontsize=9)

zustand_keep = ['Renovierungsbedarf','Gepflegt','Saniert','Vollsaniert','Modernisiert','Wie neu','Erstbezug nach Sanierung','Erstbezug']
df_zust = df[df['objektzustand_de'].isin(zustand_keep)].copy()
zust_means = df_zust.groupby('objektzustand_de')['rent_sqm'].mean().reindex(zustand_keep)
colors_z = plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(zustand_keep)))
bars2 = axes[1].barh(range(len(zust_means)), zust_means.values, color=colors_z, alpha=0.9, edgecolor='white')
axes[1].set_yticks(range(len(zust_means))); axes[1].set_yticklabels(zustand_keep, fontsize=10)
axes[1].set_xlabel('Ø Kaltmiete je m² (€)'); axes[1].set_title('Mietpreis nach Objektzustand')
axes[1].axvline(df['rent_sqm'].mean(), color=PALETTE_ACC, linestyle='--', linewidth=1.5, label='Gesamtmittel')
axes[1].legend()
for bar, v in zip(bars2, zust_means.values): axes[1].text(v+0.02, bar.get_y()+bar.get_height()/2, f'{v:.2f}€', va='center', fontsize=9)

fig.suptitle('Qualitätsmerkmale und ihr Einfluss auf den Quadratmeterpreis', fontsize=14, fontweight='bold')
plt.tight_layout(); savefig('06_ausstattung_zustand')

### 7.6 Bauepoche und Mietpreis

Gegen die intuitive Erwartung erzielen Gründerzeitbauten (<1919) und Nachkriegsgebäude höhere Quadratmeterpreise als viele neuere Gebäude. Das erklärt sich durch die innenstadtnahe Lage dieser Altbaubestände und die oft großzügigeren Grundrisse.

In [ ]:
epoche_order = ['Industrialisierung (<1919)','Zwischenkriegszeit (1919–39)','(Nach-)kriegszeit (1940-1959)',
                'Kohle und Stahlkrise (1960–89)','Post-industrieller Strukturwandel (1990–2009)','Modern (2009+)','Unbekannt']
df_ep = df[df['bauepoche'] != 'Unbekannt'].copy()
epoche_stats = df_ep.groupby('bauepoche')['rent_sqm'].agg(['mean','median','count','std'])\
                    .reindex([e for e in epoche_order if e in df_ep['bauepoche'].unique()])

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(epoche_stats))
colors_e = plt.cm.viridis(np.linspace(0.15, 0.85, len(epoche_stats)))
bars = ax.bar(x, epoche_stats['mean'], color=colors_e, alpha=0.95, edgecolor='white', width=0.6)
ax.axhline(df['rent_sqm'].mean(), color=PALETTE_NEG, linestyle='--', linewidth=1.5,
           label=f"Gesamtmittel: {df['rent_sqm'].mean():.2f} €/m²")
ax.set_xticks(x); ax.set_xticklabels(epoche_stats.index, ha='center', fontsize=7)
ax.set_ylabel('Ø Kaltmiete je m² (€)')
ax.set_title('Mietpreis nach Bauepoche – Historische Entwicklung des Wohnungsbestands')
ax.legend()
for bar, (_, row) in zip(bars, epoche_stats.iterrows()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.04,
            f"{row['mean']:.2f}€\n(n={row['count']:,.0f})", ha='center', va='bottom', fontsize=14)
plt.tight_layout(); savefig('07_bauepoche')

### 7.7 Wohnungstypenprofil nach Stadtteil

Die Stapelung zeigt auf einem Blick, welche Stadtteile von kleinen Wohnungen (Bedarf: ÖPNV, grüne Infrastruktur) vs. Familienwohnungen (Bedarf: Schulen, Parks) dominiert werden — ein direkter Steuerungshinweis für die Stadtverwaltung.

In [ ]:
df['stadtteil'].unique()

In [ ]:
df_st = df[df['plz'].notna()].copy()
zimmer_st = df_st.groupby('plz')['zimmeranzahl'].value_counts(normalize=True).unstack(fill_value=0)
zimmer_simple = pd.DataFrame({
    '1 Zi.': zimmer_st.get(1.0,0) + zimmer_st.get(1.5,0),
    '2 Zi.': zimmer_st.get(2.0,0) + zimmer_st.get(2.5,0),
    '3 Zi.': zimmer_st.get(3.0,0) + zimmer_st.get(3.5,0),
    '4+ Zi.': sum(zimmer_st.get(c,0) for c in [4.0,4.5,5.0,5.5,6.0])
})
obs_st = df_st.groupby('plz').size()
valid_st = obs_st[obs_st >= 200].index
zimmer_simple = zimmer_simple.loc[zimmer_simple.index.isin(valid_st)].sort_values('1 Zi.', ascending=True)

fig, ax = plt.subplots(figsize=(14, max(6, len(zimmer_simple)*0.45)))
colors_z4 = [PALETTE_NEG, PALETTE_ACC, PALETTE_MAIN, PALETTE_POS]
zimmer_simple.plot(kind='barh', stacked=True, ax=ax, color=colors_z4, alpha=0.9, edgecolor='white', linewidth=0.3)
ax.set_xlabel('Anteil der Wohnungstypen')
ax.set_ylabel('PLZ')
ax.set_title('Wohnungstypenprofil nach PLZ\n(Anteil an Gesamtangebot)')
ax.legend(loc='lower right', frameon=True)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.0%}'))
plt.tight_layout(); savefig('08_zimmer_stadtteil')

### 7.8 Energieeffizienz und Betriebskosten

Hocheffizienz-Gebäude (A+, A) erzielen erwartungsgemäß höhere Kaltmieten, da sie als modernere Neubauten mit besserer Ausstattung am Markt angeboten werden. Gleichzeitig sollten die Heizkosten bei schlechterer Energieklasse steigen — dieser Zusammenhang ist im Datensatz aufgrund der hohen Missingsrate bei den Heizkosten nur begrenzt messbar.

In [ ]:
eff_order = ['APLUS','A','B','C','D','E','F','G','H']
df_eff = df[df['energieeffizienzklasse'].isin(eff_order)].copy()
eff_stats = df_eff.groupby('energieeffizienzklasse').agg(
    rent_mean=('rent_sqm','mean'), heizk_mean=('heizk_sqm','mean'), count=('obid','size')
).reindex([e for e in eff_order if e in df_eff['energieeffizienzklasse'].unique()])

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
colors_eff = plt.cm.RdYlGn(np.linspace(0.9, 0.1, len(eff_stats)))
axes[0].bar(eff_stats.index, eff_stats['rent_mean'], color=colors_eff, alpha=0.9, edgecolor='white')
axes[0].axhline(df['rent_sqm'].mean(), color=PALETTE_MAIN, linestyle='--', linewidth=1.5, label='Gesamtmittel')
axes[0].set_xlabel('Energieeffizienzklasse'); axes[0].set_ylabel('Ø Kaltmiete je m² (€)')
axes[0].set_title('Kaltmiete nach Energieeffizienzklasse'); axes[0].legend()
for i,(idx,row) in enumerate(eff_stats.iterrows()):
    axes[0].text(i, row['rent_mean']+0.05, f"{row['rent_mean']:.2f}€\n(n={row['count']:,})", ha='center', va='bottom', fontsize=8)

eff_hk = eff_stats.dropna(subset=['heizk_mean'])
axes[1].bar(eff_hk.index, eff_hk['heizk_mean'], color=colors_eff[:len(eff_hk)], alpha=0.9, edgecolor='white')
axes[1].set_xlabel('Energieeffizienzklasse'); axes[1].set_ylabel('Ø Heizkosten je m² (€)')
axes[1].set_title('Heizkosten nach Energieeffizienzklasse')

fig.suptitle('Energieeffizienz – Kaltmiete und Betriebskosten im Vergleich', fontsize=14, fontweight='bold')
plt.tight_layout(); savefig('09_energieeffizienz')

### 7.9 Streudiagramm – Wohnfläche und Mietpreis nach Ausstattung

Ein nicht vorhandener (weil leicht positiver) Skaleneffekt ist gut erkennbar: größere Wohnungen haben tendenziell einen niedrigeren qm-Preis, bzw. im Falle von Dortmund ist dieser Effekt schwach und statistisch nicht signifikant ausgeprägt. Deluxe-Wohnungen streuen nach oben, was auf einen nicht-linearen Preiseffekt bei sehr hochwertiger Ausstattung hindeutet.

In [ ]:
df_sc = df[df['ausstattung'].isin(['Simple','Normal','Sophisticated','Deluxe'])].copy()
palette_sc = {'Simple':'#e74c3c','Normal':'#f39c12','Sophisticated':'#2980b9','Deluxe':'#27ae60'}

fig, ax = plt.subplots(figsize=(14, 8))
for quality, group in df_sc.groupby('ausstattung'):
    sample = group.sample(min(800, len(group)), random_state=42)
    ax.scatter(sample['wohnflaeche'], sample['rent_sqm'],
               color=palette_sc[quality], alpha=0.35, s=18, label=quality)

x_fit = np.linspace(df['wohnflaeche'].quantile(0.01), df['wohnflaeche'].quantile(0.99), 200)
slope, intercept, r, p, _ = stats.linregress(df['wohnflaeche'].dropna(), df.loc[df['wohnflaeche'].notna(),'rent_sqm'])
ax.plot(x_fit, intercept+slope*x_fit, color='black', linewidth=2, linestyle='--',
        label=f'Linearer Trend (R={r:.3f})')

ax.set_xlabel('Wohnfläche (m²)'); ax.set_ylabel('Kaltmiete je m² (€)')
ax.set_title('Wohnfläche und Mietpreis nach Ausstattungsqualität')
ax.legend(markerscale=2, frameon=True)
plt.tight_layout(); savefig('10_scatter_flaeche_miete')

## 8. Multivariante Regressionsanalyse

### Schrittweiser Modellaufbau (OLS)

| Modell | Variablen | adj. R² |
|--------|-----------|----------|
| Modell 1 | Fläche + Zimmer | ~0.025 |
| Modell 2 | + Ausstattung + Zustand | ~0.273 |
| Modell 3 | + Nebenkosten + Modernisierung | ~0.355 |
| Modell 4 | + Ausstattungsfeatures + Energieeffizienz | **~0.641** |

Modell 4 erklärt 64% der Varianz im qm-Preis. Der stärkste Einzeleffekt ist die **Ausstattungsqualität** (+1,13 €/m² je Stufe), gefolgt vom **Aufzug** (+0,91 €/m²). Überraschend negativ wirkt sich das Merkmal **Modernisiert** aus (−0,68 €/m²), was als Selektionseffekt interpretiert werden kann: Wohnungen mit explizit kommunizierter Modernisierung stammen häufig aus dem unteren bis mittleren Preissegment.

In [ ]:
list(df)

In [ ]:
# Regressionsdatensatz
reg_cols = [
    'rent_sqm','wohnflaeche','zimmeranzahl','gebaeudealter',
    'ausstattung_num','zustand_num','heizk_sqm','nk_sqm',
    'wurde_modernisiert','balkon_bin','aufzug_bin','einbaukueche_bin',
    'keller_bin','gaestewc_bin','parkplatz_bin','eeff_num','plz'
]
df_reg = df[reg_cols].copy().dropna(subset=['rent_sqm','wohnflaeche','zimmeranzahl'])
df_reg = df_reg[df_reg['heizk_sqm'] < 5].copy()
print(f'Regressions-Stichprobe: {len(df_reg):,} Beobachtungen')

In [ ]:
# Modell 1: Basis
m1 = smf.ols('rent_sqm ~ wohnflaeche + zimmeranzahl', data=df_reg).fit()
print(f'Modell 1 – adj. R² = {m1.rsquared_adj:.4f}')
print(m1.summary())

In [ ]:
# Modell 2: + Qualität
m2 = smf.ols('rent_sqm ~ wohnflaeche + zimmeranzahl + ausstattung_num + zustand_num',
             data=df_reg.dropna(subset=['ausstattung_num','zustand_num'])).fit()
print(f'Modell 2 – adj. R² = {m2.rsquared_adj:.4f}')
print(m2.summary())

In [ ]:
# Modell 3: + Kosten & Modernisierung
m3_data = df_reg.dropna(subset=['ausstattung_num','zustand_num','heizk_sqm','nk_sqm'])
m3 = smf.ols('rent_sqm ~ wohnflaeche + zimmeranzahl + ausstattung_num + zustand_num '
             '+ heizk_sqm + nk_sqm + wurde_modernisiert', data=m3_data).fit()
print(f'Modell 3 – adj. R² = {m3.rsquared_adj:.4f}')
print(m3.summary2())

In [ ]:
# Modell 4: Vollmodell
m4_data = df_reg.dropna(subset=['ausstattung_num','zustand_num','heizk_sqm','nk_sqm','eeff_num'])
m4 = smf.ols(
    'rent_sqm ~ wohnflaeche + zimmeranzahl + ausstattung_num + zustand_num '
    '+ heizk_sqm + nk_sqm + wurde_modernisiert '
    '+ balkon_bin + aufzug_bin + einbaukueche_bin + keller_bin + gaestewc_bin '
    '+ parkplatz_bin + eeff_num',
    data=m4_data
).fit()
print(m4.summary())

### 8.1 Koeffizientenplot – Modell 4

Der Plot zeigt Richtung und Stärke aller Effekte mit 95%-Konfidenzintervallen. Sterne kennzeichnen das Signifikanzniveau (*** p<0.001, ** p<0.01, * p<0.05).

In [ ]:
coef_df = pd.DataFrame({
    'coef': m4.params, 'ci_lo': m4.conf_int()[0],
    'ci_hi': m4.conf_int()[1], 'pval': m4.pvalues
}).drop(index='Intercept')

label_map = {
    'wohnflaeche':'Wohnfläche (m²)', 'zimmeranzahl':'Zimmeranzahl',
    'ausstattung_num':'Ausstattungsqualität (1–4)', 'zustand_num':'Objektzustand (Ordinal)',
    'heizk_sqm':'Heizkosten je m²', 'nk_sqm':'Nebenkosten je m²',
    'wurde_modernisiert':'Modernisiert (Dummy)', 'balkon_bin':'Balkon',
    'aufzug_bin':'Aufzug', 'einbaukueche_bin':'Einbauküche',
    'keller_bin':'Keller', 'gaestewc_bin':'Gäste-WC',
    'parkplatz_bin':'Parkplatz', 'eeff_num':'Energieeffizienz (1=A+, 9=H)',
}
coef_df.index = [label_map.get(i,i) for i in coef_df.index]
coef_df = coef_df.sort_values('coef')

fig, ax = plt.subplots(figsize=(12, 8))
colors_coef = [PALETTE_POS if c > 0 else PALETTE_NEG for c in coef_df['coef']]
ax.barh(range(len(coef_df)), coef_df['coef'], color=colors_coef, alpha=0.85, edgecolor='white', height=0.6)
ax.errorbar(coef_df['coef'], range(len(coef_df)),
            xerr=[coef_df['coef']-coef_df['ci_lo'], coef_df['ci_hi']-coef_df['coef']],
            fmt='none', color='black', capsize=4, linewidth=1.5)
ax.axvline(0, color='black', linewidth=1.2)
ax.set_yticks(range(len(coef_df))); ax.set_yticklabels(coef_df.index, fontsize=10)
ax.set_xlabel('Regressionskoeffizient (€/m²-Effekt)')
ax.set_title(f'Koeffizientenplot – Vollmodell OLS\nadj. R² = {m4.rsquared_adj:.3f}  |  n = {int(m4.nobs):,}')
for i,(_, row) in enumerate(coef_df.iterrows()):
    stars = '***' if row['pval']<0.001 else '**' if row['pval']<0.01 else '*' if row['pval']<0.05 else ''
    if stars:
        xpos = row['ci_hi']+0.003 if row['coef']>=0 else row['ci_lo']-0.003
        ax.text(xpos, i, stars, va='center', fontsize=9, ha='left' if row['coef']>=0 else 'right')
plt.tight_layout(); savefig('12_koeffizientenplot')

### 8.2 Korrelationsmatrix

Die Paarweise Korrelationsmatrix aller Regressionsvariablen zeigt Multikollinearitätsrisiken und stärkt die inhaltliche Interpretation der Modellkoeffizienten.

In [ ]:
corr_cols_label = {
    'rent_sqm':'Kaltmiete je m²', 'wohnflaeche':'Wohnfläche', 'zimmeranzahl':'Zimmeranzahl',
    'gebaeudealter':'Gebäudealter', 'ausstattung_num':'Ausstattung', 'zustand_num':'Objektzustand',
    'heizk_sqm':'Heizkosten/m²', 'nk_sqm':'Nebenkosten/m²', 'wurde_modernisiert':'Modernisiert',
    'balkon_bin':'Balkon', 'aufzug_bin':'Aufzug', 'einbaukueche_bin':'Einbauküche',
    'keller_bin':'Keller', 'parkplatz_bin':'Parkplatz', 'eeff_num':'Energieeffizienz',
}
df_corr = df_reg[list(corr_cols_label.keys())].copy()
df_corr.columns = list(corr_cols_label.values())
df_corr = df_corr.dropna()
corr_matrix = df_corr.corr()

fig, ax = plt.subplots(figsize=(15, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            vmin=-1, vmax=1, ax=ax, linewidths=0.5, linecolor='white',
            annot_kws={'size':9}, square=True,
            cbar_kws={'shrink':0.8,'label':'Pearson-Korrelationskoeffizient r'})
ax.set_title('Korrelationsmatrix – Immobilienqualität und Mietpreis\nDortmunder Angebotsmarkt',
             fontsize=14, fontweight='bold', pad=20)
ax.tick_params(axis='x', rotation=45, labelsize=10)
ax.tick_params(axis='y', rotation=0, labelsize=10)
plt.tight_layout(); savefig('13_korrelationsmatrix')

### 8.3 Modellvergleich und Residualanalyse

In [ ]:
# Modellvergleich
models_r2 = {
    'Modell 1\nFläche + Zimmer': m1.rsquared_adj,
    'Modell 2\n+ Qualität': m2.rsquared_adj,
    'Modell 3\n+ Kosten & Mod.': m3.rsquared_adj,
    'Modell 4\nVollmodell': m4.rsquared_adj,
}
fig, ax = plt.subplots(figsize=(12, 6))
colors_r2 = plt.cm.Blues(np.linspace(0.4, 0.9, len(models_r2)))
bars_r2 = ax.bar(list(models_r2.keys()), list(models_r2.values()),
                  color=colors_r2, alpha=0.9, edgecolor='white', width=0.5)
ax.set_ylabel('Adjustiertes R²')
ax.set_title('Schrittweiser Modellaufbau – Erklärungskraft der OLS-Modelle')
ax.set_ylim(0, min(1, max(models_r2.values())*1.25))
for bar, val in zip(bars_r2, models_r2.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f'R² = {val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.tight_layout(); savefig('14_modellvergleich')

In [ ]:
# Residualanalyse
residuals = m4.resid; fitted = m4.fittedvalues
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].scatter(fitted, residuals, alpha=0.15, s=5, color=PALETTE_MAIN)
axes[0].axhline(0, color=PALETTE_NEG, linewidth=1.5, linestyle='--')
axes[0].set_xlabel('Gefittete Werte (€/m²)'); axes[0].set_ylabel('Residuen')
axes[0].set_title('Residuen vs. gefittete Werte\n(Homoskedastizitätscheck)')
sm.qqplot(residuals, line='s', ax=axes[1], alpha=0.3, markersize=2)
axes[1].set_title('Q-Q-Plot der Residuen\n(Normalverteilungscheck)')
fig.suptitle('Diagnostik – OLS-Vollmodell (Modell 4)', fontsize=14, fontweight='bold')
plt.tight_layout(); savefig('15_residualanalyse')